<a href="https://colab.research.google.com/github/hum1-ua/Reto_Corref_PLN/blob/main/Reto2_Personajes_Sab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 2 · Reto 1 — Extracción de personajes y correferencia en *Sab*

**Procesamiento del Lenguaje Natural 1 · Grado en Ingeniería en IA · Universidad de Alicante**

**Equipo:**

- Hugo Urbán Martínez
- José Francisco Pérez Monpeán
- Gabriel Niculescu Ruso
- Patricia Pérez Ferre

## Qué hace este cuaderno

Dada la novela *Sab* (Gómez de Avellaneda, 1841, ELTeC-spa, XML-TEI), el cuaderno **marca en XML cada mención directa y explícita de un personaje** con `<rs type="person" ref="#prXXX">…</rs>`, de modo que todas las menciones al mismo personaje comparten el mismo identificador (correferencia). La salida es el mismo XML de entrada, byte a byte, salvo por las etiquetas `<rs>` insertadas.

## Resumen de la aproximación

El problema combina **NER de tipo PERSONA** y **resolución de correferencia**. Hemos optado por un sistema **híbrido, simbólico + modelo estadístico**, en lugar de un modelo de correferencia neuronal, por tres razones que justificamos más abajo: (1) no existe un modelo de correferencia robusto para español literario del siglo XIX; (2) el reparto de personajes es **un conjunto cerrado y conocido** (`listPerson.xml`), por lo que el problema es más de *entity linking* que de *clustering* libre; (3) la correferencia en narrativa depende mucho de la estructura del diálogo (quién habla a quién), que un modelo entrenado sobre prensa no captura.

| Fase | Qué hace | Herramienta |
|---|---|---|
| 0. Lectura | Extrae los párrafos de cada capítulo conservando la posición exacta en el XML original | regex + mapa de posiciones |
| 1. Análisis | Lematización, POS, dependencias, NER | spaCy `es_core_news_lg` |
| 2. Menciones | Nombres propios (gazetario + NER), SN definidos (léxico de descriptores), pronombres tónicos, colectivos («ambos») | reglas sobre el análisis de spaCy |
| 3. Diálogo | Segmenta habla/narración, identifica **hablante e interlocutor** de cada turno | reglas sobre guiones y verbos de habla |
| 4. Correferencia | Asigna cada mención a un personaje con un modelo de **saliencia** (estilo Lappin & Leass) + léxico + relaciones de parentesco | puntuación explícita |
| 5. Salida | Inserta `<rs>` en el XML original comprobando que sigue bien formado | — |
| 6. Evaluación | Compara con los capítulos 1 y 13 anotados a mano | P/R/F1 |

## 0. Instalación y configuración

In [ ]:
!pip -q install spacy lxml
!python -m spacy download es_core_news_lg -q

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re, html, math, collections, urllib.request, os
from dataclasses import dataclass, field
import spacy
from lxml import etree

# ============ CONFIGURACIÓN ============
APELLIDOS = "UrbánMartínez_PérezFerre_NiculescuRuso_PérezMonpeán"
BASE_RAW = "https://raw.githubusercontent.com/"
URL_NOVELA = BASE_RAW + "COST-ELTeC/ELTeC-spa/refs/heads/master/level1/SPA1021_GomezDeAvellaneda_Sab.xml"
URL_PERSONAJES = BASE_RAW + "jaspock/plni-2627/refs/heads/main/docs/assets/resources/listPerson.xml"
URL_GOLD = {
    1:  BASE_RAW + "jaspock/plni-2627/refs/heads/main/docs/assets/resources/GomezDeAvellaneda_Sab_Cap01_AnotacionPersonajes.xml",
    13: BASE_RAW + "jaspock/plni-2627/refs/heads/main/docs/assets/resources/GomezDeAvellaneda_Sab_Cap13_AnotacionPersonajes.xml",
}
DEBUG = bool(os.environ.get("DEBUG_DIALOGO"))
FICHERO_SALIDA = f"SPA1021_GomezDeAvellaneda_Sab_Anotado{APELLIDOS}.xml"

def leer(origen):
    """Lee una URL o un fichero local y devuelve el texto."""
    if origen.startswith("http"):
        with urllib.request.urlopen(origen) as r:
            return r.read().decode("utf-8")
    with open(origen, encoding="utf-8") as f:
        return f.read()

## 1. Datos: novela y reparto de personajes

Cargamos la novela y la lista de personajes. **Los identificadores se leen del fichero `listPerson.xml`** (no están escritos a mano en el código), para usar exactamente los oficiales.

In [ ]:
XML_NOVELA = leer(URL_NOVELA)
LISTA_PERS = leer(URL_PERSONAJES)

ids_oficiales = re.findall(r'xml:id="([^"]+)"', LISTA_PERS)
nombres_oficiales = re.findall(r"<persName>(.*?)</persName>", LISTA_PERS)
print(len(ids_oficiales), "personajes:", list(zip(ids_oficiales, nombres_oficiales))[:6], "...")

21 personajes: [('pr001', 'Sab (Bernabé)'), ('pr002', 'Carlota de B...'), ('pr003', 'Enrique Otway'), ('pr004', 'Teresa'), ('pr005', 'Don Carlos de B...'), ('pr006', 'Jorge Otway')] ...


### 1.1 Conocimiento sobre el reparto

Aunque *Sab* es una sola novela, parte del conocimiento que usamos **no es específico de ella**: la gramática de los pronombres, los verbos de habla, la estructura de guiones del diálogo o la idea de «parentesco» son generales. Lo único específico de *Sab* es esta tabla (género de cada personaje, cómo se le nombra y cómo se le describe) y las relaciones de parentesco, que son datos de la propia novela (se pueden extraer de la lista de personajes y de la sinopsis).

Los personajes se numeran del 1 al 21 en el mismo orden que `listPerson.xml`:
1 Sab · 2 Carlota · 3 Enrique Otway · 4 Teresa · 5 don Carlos · 6 Jorge Otway · 7 don Luis · 8 don Félix · 9 don Agustín · 10 Eugenio · 11 Martina · 12 madre de Sab · 13 madre de Carlota · 14-17 hermanas de Carlota · 18 Belén · 19 los dos catalanes (plural) · 20 José · 21 esclavo sin nombre.

In [ ]:
GEN = {1:'m',2:'f',3:'m',4:'f',5:'m',6:'m',7:'m',8:'m',9:'m',10:'m',11:'f',12:'f',13:'f',
       14:'f',15:'f',16:'f',17:'f',18:'f',19:'m',20:'m',21:'m'}
PLURAL = {19}                       # los catalanes: no se anotan menciones singulares

# Nombres propios: (personaje, expresión regular). Los títulos (don, D.) y el complemento «de B...» entran en la mención.
T = r"(?:(?:[Dd]on|D\.|Dn\.)\s*)"          # título honorífico masculino
TF = r"(?:(?:[Dd]oña|Dña\.|D\.ª)\s*)"
DB = r"(?:\s+de\s+B\.\.\.)?"                              # «de B...»
NOMBRES = [
    (1,  r"Sab"), (1, r"Bernabé"),
    (2,  r"Carlota" + DB),
    (3,  r"Enrique\s+Otway"), (3, r"Enrique"),
    (4,  r"Teresa"),
    (5,  T + r"Carlos" + DB), (5, r"Carlos"),
    (6,  r"Jorge\s+Otway"), (6, r"Jorge"),
    (7,  T + r"Luis" + DB), (7, r"Luis"),
    (8,  T + r"Félix" + DB), (8, r"Félix"),
    (9,  T + r"Agustín" + DB), (9, r"Agustín"),
    (10, r"Eugenio" + DB),
    (11, r"Martina"),
    (18, r"Belén"),
    (20, r"José"),
    # perífrasis fijas con «de B...»
    (2,  r"(?:[Ll]a\s+)?señorita\s+de\s+B\.\.\."),
    (5,  r"(?:[Ee]l\s+)?señor\s+de\s+B\.\.\."),
    (13, r"(?:[Ll]a\s+)?señora\s+de\s+B\.\.\."),
]
# «Otway» suelto es ambiguo (Enrique o su padre Jorge)
RE_OTWAY = re.compile(r"(?<!\w)Otway(?!\w)")
RE_OTWAY_PADRE = re.compile(r"(?:viejo|padre|anciano|Jorge)\s+$", re.I)

# Descriptores (sustantivo/adjetivo sustantivado -> {personaje: peso}). 1 = identifica casi seguro.
DESC = {
    # Sab
    "mulato":{1:1}, "campesino":{1:1}, "labriego":{1:1}, "paisano":{1:1}, "guía":{1:1}, "mayoral":{1:1},
    "esclavo":{1:.6,20:.6,21:.6}, "negro":{21:1,20:.3},
    # Carlota
    "señorita":{2:1}, "ángel":{2:.8}, "niña":{2:.5}, "criatura":{2:.5}, "hechicera":{2:.5},
    # Enrique
    "extranjero":{3:1}, "viajero":{3:1}, "forastero":{3:1}, "inglés":{3:.7,6:.3}, "amante":{3:1},
    "prometido":{3:1}, "novio":{3:1}, "recién llegado":{3:1}, "interlocutor":{3:.5,1:.5},
    # Teresa
    "doncella":{4:1}, "huérfana":{4:1}, "señora":{4:.5,13:.4},
    # don Carlos, don Luis, Jorge, Martina, Belén
    "dueño":{5:1}, "amo":{5:1}, "propietario":{5:1}, "bienhechor":{5:.8,7:.2},
    "protector":{7:1}, "difunto":{7:.6},
    "viejo":{6:.6,11:.5,5:.2}, "anciano":{6:.5,5:.2}, "india":{11:1}, "vieja":{11:1}, "esclava":{18:1},
    # genéricos con apoyo en el género
    "joven":{3:.6,1:.3,7:.3,2:.3,4:.2}, "caballero":{3:.3,5:.4,7:.2},
}
# Sustantivos de persona que, con determinante definido/demostrativo, pueden referirse anafóricamente a un personaje saliente
GENERICOS = {"hombre","joven","mujer","señor","señora","caballero","viejo","vieja","anciano","infeliz","miserable",
             "mentecato","bruto","desgraciado","desdichado","amigo","amiga","criatura","ángel","huérfana",
             "extranjero","esclavo","esclava","negro","mulato","doncella","señorita","niña","protector","amante",
             "ídolo","pobre"}
# Términos de parentesco/relación: se resuelven según el «poseedor»
REL = {"padre":"padre","madre":"madre","hijo":"hijo","hija":"hija","hermano":"hermano","hermana":"hermana",
       "amo":"amo","dueño":"amo","señor":"amo","bienhechor":"amo"}
KIN = {("padre",2):5, ("padre",1):7, ("padre",3):6, ("padre",7):8, ("padre",5):8,
       ("madre",2):13, ("madre",1):12,
       ("hijo",6):3, ("hijo",7):1, ("hijo",12):1, ("hijo",11):1, ("hija",5):2, ("hija",13):2,
       ("hermano",7):5, ("hermano",5):7,
       ("amo",1):5, ("amo",20):5, ("amo",21):5, ("amo",18):5, ("amo",11):5}
# Prior global (frecuencia aproximada en la novela) para desempatar cuando no hay saliencia
PRIOR = {1:.30, 2:.25, 3:.20, 4:.15, 5:.10, 6:.05, 7:.05}

VERBOS_HABLA = re.compile(r"^(dij|díj|exclam|añad|respond|repus|replic|pregunt|contest|murmur|repit|continu|prosigui|"
                          r"interrump|grit|suspir|decía|observ|agreg|manifest|concluy|insist|balbuce|sollo|clam|"
                          r"prorrump|susurr|llam|tartamude|volvi[oó]|dij)", re.I)

# Pronombres personales tónicos: forma -> (persona, género, número)
PRON = {"yo":(1,None,'s'), "mí":(1,None,'s'), "tú":(2,None,'s'), "ti":(2,None,'s'), "vos":(2,None,'s'),
        "usted":(2,None,'s'), "vd":(2,None,'s'), "ud":(2,None,'s'),
        "él":(3,'m','s'), "ella":(3,'f','s'), "aquél":(3,'m','s'), "aquélla":(3,'f','s')}
RE_PRON = re.compile(r"(?<![\wÀ-ÿ])(yo|mí|tú|ti|vos|usted|Vd\.|Ud\.|Vd|Ud|él|ella|aquél|aquélla)"
                     r"(\s+mism[oa])?(?![\wÀ-ÿ])", re.I)
RE_COLECTIVO = re.compile(r"(?<![\wÀ-ÿ])(ambos|los dos)(?![\wÀ-ÿ])", re.I)

## 2. Lectura del XML conservando las posiciones

Requisito clave: la salida debe ser **el mismo XML que la entrada** salvo por las menciones. Por eso **no reserializamos el árbol** (lxml cambiaría espacios, entidades o el orden de atributos): trabajamos sobre el texto crudo del fichero, localizamos cada `<p>` de cada capítulo y, para cada párrafo, construimos:
* el **texto plano** (sin etiquetas, con las entidades `&amp;` decodificadas y el contenido de `<note>` ignorado), que es lo que analiza spaCy;
* un **mapa carácter → posición en el XML**, para poder insertar `<rs>` y `</rs>` en el sitio exacto;
* el **elemento interno** de cada carácter (p. ej. un `<emph>`), para no abrir un `<rs>` dentro de un elemento y cerrarlo fuera (XML mal formado).

Solo se procesan los `<p>` del cuerpo de cada capítulo (no los epígrafes en `<quote>`, ni `<head>`, ni los metadatos).

In [ ]:
RE_TOK = re.compile(r"(<[^>]*>)|(&[#\w]+;)|([^<&]+)|(&)")
MASK = {"note"}

def aplana(interior):
    """Devuelve (texto, ini_xml, fin_xml, elemento, spans_rs). ini/fin: posición XML de cada carácter;
    elemento: id del elemento más interno que lo contiene; spans_rs: <rs> ya existentes (para leer el gold)."""
    txt, ini, fin, elem = [], [], [], []
    pila, rs_abiertos, rs_spans, uid, mask = [], [], [], 0, 0
    for m in RE_TOK.finditer(interior):
        tag, ent, plano, amp = m.groups()
        if tag:
            nombre = re.match(r"</?\s*([\w:.-]+)", tag)
            nombre = nombre.group(1) if nombre else ""
            if tag.startswith("</"):
                if nombre == "rs":
                    if rs_abiertos:
                        s, ref = rs_abiertos.pop(); rs_spans.append((s, len(txt), ref))
                elif pila:
                    pila.pop()
                    if nombre in MASK: mask -= 1
            elif not tag.endswith("/>"):
                if nombre == "rs":
                    r = re.search(r'ref="([^"]*)"', tag); rs_abiertos.append((len(txt), r.group(1) if r else ""))
                else:
                    uid += 1; pila.append(uid)
                    if nombre in MASK: mask += 1
            continue
        if mask: continue
        el = pila[-1] if pila else 0
        if ent:
            txt.append(html.unescape(ent)); ini.append(m.start()); fin.append(m.end()); elem.append(el)
        else:
            for k, ch in enumerate(plano or amp):
                txt.append(ch); ini.append(m.start() + k); fin.append(m.start() + k + 1); elem.append(el)
    return "".join(txt), ini, fin, elem, rs_spans

def capitulos(xml):
    """Itera (numero, inicio, fin) de cada <div type="chapter"> en el texto crudo (sin anidamiento de divs)."""
    for m in re.finditer(r'<div type="chapter" n="SPA1021(\d{3})">', xml):
        depth, pos = 1, m.end()
        for t in re.finditer(r"<div[ >]|</div>", xml[pos:]):
            depth += 1 if t.group().startswith("<div") else -1
            if depth == 0:
                yield int(m.group(1)), m.start(), pos + t.end(); break

def parrafos(xml, ini, fin):
    """Párrafos <p> de un capítulo que no están en <quote> ni <note>. Devuelve (ini_interior, fin_interior)."""
    sub = xml[ini:fin]
    excl = [(a.start(), a.end()) for a in re.finditer(r"<(quote|note)\b.*?</\1>", sub, flags=re.S)]
    for p in re.finditer(r"<p(?:\s[^>]*)?>(.*?)</p>", sub, flags=re.S):
        if any(a <= p.start() < b for a, b in excl): continue
        yield ini + p.start(1), ini + p.end(1)

caps = {n: (a, b) for n, a, b in capitulos(XML_NOVELA)}
print(len(caps), "capítulos;", sum(1 for n in caps for _ in parrafos(XML_NOVELA, *caps[n])), "párrafos de texto")

17 capítulos; 896 párrafos de texto


## 3. Análisis lingüístico y detección de menciones

Cargamos spaCy (`es_core_news_lg`: lematizador, etiquetador, parser de dependencias y NER entrenado en AnCora/WikiNER). Una **mención** es un fragmento del texto con su tipo y sus rasgos:

* **Nombre propio** (`nombre`): gazetario del reparto (con título y «de B...») y, como red de seguridad, las entidades `PER` de spaCy que se parecen a un nombre del reparto (variantes y diminutivos: *Carlotica*, *Sabito*…). Un nombre propio **no necesita desambiguación** (salvo *Otway*).
* **Sintagma nominal** (`np`): núcleo nominal de persona + determinantes y adjetivos a su izquierda (*el joven*, *ese ángel de tus ilusiones*, *mi amo actual*), más el complemento «de + nombre» en las relaciones (*padre de Carlota*). Según el determinante: **definido/demostrativo/posesivo** (se resuelve por saliencia), **indefinido** (solo se anota la primera presentación del personaje, como indica la guía) o **sin determinante** (en el habla es un *vocativo*: *«señora»*, *«buen amigo»*).
* **Pronombre tónico** (`pron`): *yo, tú, vos, usted, él, ella, mí, ti…* (con *mismo/a*). Los átonos quedan fuera, como pide el enunciado.
* **Colectivo** (`colectivo`): *ambos*, *los dos*: remiten a **dos** personajes (`ref="#pr004 #pr001"`, como en el gold).

In [ ]:
nlp = None
for modelo in ("es_core_news_lg", "es_core_news_md", "es_core_news_sm"):
    try:
        nlp = spacy.load(modelo); print("Modelo spaCy:", modelo); break
    except OSError:
        continue

@dataclass
class Mencion:
    ini: int; fin: int; tipo: str            # 'nombre' | 'np' | 'pron' | 'colectivo'
    texto: str = ""
    cand: dict = field(default_factory=dict)  # {personaje: peso} para nombres/NP con descriptor
    persona: int = 3; genero: str = None
    det: str = ""                             # 'def' | 'indef' | 'none' | 'pos_mi' | 'pos_tu' | 'pos_su'
    lema: str = ""; sujeto: bool = False; oracion: int = 0
    pos_de: int = None                        # poseedor explícito «padre de Carlota»
    prev: str = ""; sig: str = ""             # palabra anterior y signo siguiente (para detectar vocativos)
    ref: list = None                          # personajes asignados (resultado)

SOLO_NOMBRE = {"negro", "viejo", "joven", "inglés"}

def lemas_rel(tok):
    for k in (tok.text.lower(), tok.lemma_.lower()):
        if k in REL: return REL[k]
    return None

def clave_lexico(tok):
    for k in (tok.lemma_.lower(), tok.text.lower()):
        if k in DESC or k in GENERICOS or k in REL: return k
    return None

def menciones_nombre(texto):
    """Nombres propios por gazetario; devuelve lista de Mencion."""
    out = []
    for pid, rx in NOMBRES:
        for m in re.finditer(r"(?<![\wÀ-ÿ])" + rx + r"(?![\wÀ-ÿ])", texto):
            out.append(Mencion(m.start(), m.end(), "nombre", m.group(), cand={pid: 1.0}, genero=GEN[pid]))
    for m in RE_OTWAY.finditer(texto):
        antes = texto[max(0, m.start() - 12):m.start()]
        pid = 6 if RE_OTWAY_PADRE.search(antes) else 3
        out.append(Mencion(m.start(), m.end(), "nombre", m.group(), cand={pid: 1.0}, genero="m"))
    return out

def similar(a, b):
    import difflib
    a, b = a.lower(), b.lower()
    for suf in ("ito","ita","ico","ica","illo","illa"):
        if a.endswith(suf): a = a[:-len(suf)]
    return difflib.SequenceMatcher(None, a, b[:max(len(a),3)]).ratio()

def menciones_ner(doc, ocupado):
    """Entidades PER de spaCy no cubiertas por el gazetario, enlazadas por parecido con un nombre del reparto."""
    out = []
    simples = {1:"Sab",2:"Carlota",3:"Enrique",4:"Teresa",6:"Jorge",11:"Martina",18:"Belén",20:"José"}
    for e in doc.ents:
        if e.label_ != "PER": continue
        if any(not (e.end_char <= a or e.start_char >= b) for a, b in ocupado): continue
        mejor = max(((similar(e.text.split()[-1], n), pid) for pid, n in simples.items()), default=(0, None))
        if mejor[0] >= .8:
            out.append(Mencion(e.start_char, e.end_char, "nombre", e.text, cand={mejor[1]: .9}, genero=GEN[mejor[1]]))
    return out

def menciones_pron(texto):
    out = []
    for m in RE_PRON.finditer(texto):
        base = m.group(1).lower().rstrip(".")
        p, g, n = PRON[base]
        out.append(Mencion(m.start(), m.end(), "pron", m.group(), persona=p, genero=g, lema=base))
    for m in RE_COLECTIVO.finditer(texto):
        out.append(Mencion(m.start(), m.end(), "colectivo", m.group()))
    return out

DET_DEF = {"el","la","este","esta","ese","esa","aquel","aquella"}
DET_POS = {"mi":"pos_mi", "tu":"pos_tu", "su":"pos_su", "vuestro":"pos_tu", "vuestra":"pos_tu", "nuestro":"pos_su", "nuestra":"pos_su"}
DET_INDEF = {"un", "una"}
DET_PLUR = {"los","las","unos","unas","mis","tus","sus","estos","esos","aquellos","vuestros","nuestros","todos","estas","esas","aquellas"}
DET_ALL = DET_DEF | set(DET_POS) | DET_INDEF | DET_PLUR

def palabra(tok):
    """Texto en minúscula sin signos iniciales pegados («-Mi» -> «mi») y su posición real. spaCy confunde a veces
    posesivos con PROPN cuando van tras una coma, así que los determinantes se reconocen por lista cerrada, no por POS."""
    t = tok.text; k = len(t) - len(t.lstrip("-—–«»¡¿\"")); return t[k:].lower(), tok.idx + k

def tipo_det(w):
    return ("def" if w in DET_DEF else DET_POS.get(w) or ("indef" if w in DET_INDEF else "plural" if w in DET_PLUR else None))

def menciones_np(doc, texto):
    """SN de persona: núcleo nominal del léxico + determinantes/adjetivos a la izquierda (+ «de NOMBRE» en relaciones)."""
    out = []
    for tok in doc:
        k = clave_lexico(tok)
        if not k or tok.pos_ not in ("NOUN", "ADJ", "PROPN", "X"): continue
        if k in SOLO_NOMBRE and tok.pos_ != "NOUN": continue          # «negro» color vs. «el negro» persona
        if "Plur" in tok.morph.get("Number"): continue
        i, j, det = tok.i, tok.i, "none"
        ini_c = tok.idx
        while j - 1 >= 0 and det == "none":
            w, p0 = palabra(doc[j-1])
            if texto[doc[j-1].idx + len(doc[j-1]):doc[j].idx].strip() != "": break
            if w in ("al", "del"):                                   # contracción a+el / de+el: la mención empieza en el nombre
                det = "def"; break
            if tipo_det(w):
                det = tipo_det(w)
            elif not (doc[j-1].pos_ == "ADJ" and w.isalpha() and w not in ("sí", "no", "ya")):
                break
            j -= 1; ini_c = p0
        if det == "plural": continue
        fin_c = tok.idx + len(tok)
        # post-adjetivo breve («mi amo actual»)
        if i + 1 < len(doc) and doc[i+1].text.lower() in ("actual", "mismo", "misma") and doc[i+1].idx == fin_c + 1:
            fin_c = doc[i+1].idx + len(doc[i+1])
        # relación + «de B...» / «de NOMBRE»
        pos_de = None
        r = re.match(r"\s+de\s+(B\.\.\.|[A-ZÁÉÍÓÚ][\wÀ-ÿ]+)", texto[fin_c:])
        if r and k in REL:
            nm = r.group(1)
            for pid, rx in NOMBRES:
                if re.fullmatch(rx, nm):
                    pos_de = pid; break
            if pos_de or nm == "B...":
                fin_c += r.end()
        # indefinido con complemento descriptivo («un joven de hermosa presencia»)
        elif det == "indef":
            r = re.match(r"\s+de\s+(?:\w+\s+){0,1}\w+", texto[fin_c:])
            if r: fin_c += r.end()
        g = tok.morph.get("Gender")
        out.append(Mencion(ini_c, fin_c, "np", texto[ini_c:fin_c], cand=dict(DESC.get(k, {})),
                           genero=("m" if "Masc" in g else "f" if "Fem" in g else None), det=det, lema=k, pos_de=pos_de,
                           sujeto=tok.dep_.startswith("nsubj"), oracion=0,
                           prev=(palabra(doc[j-1])[0] if j > 0 else ""), sig=texto[fin_c:fin_c + 3].strip()[:1]))
    return out

def extiende_nombres(doc, texto, nombres):
    """«la señorita Carlota», «el viejo Otway», «mi protector don Luis»: se amplía el nombre con lo que lo modifica por la izquierda."""
    DETS = {"el","la","su","mi","tu","nuestro","este","ese","aquel","esta","esa","aquella"}
    for n in nombres:
        toks = [t for t in doc if t.idx + len(t) <= n.ini]
        j, ini, util = len(toks) - 1, n.ini, False
        while j >= 0 and texto[toks[j].idx + len(toks[j]):ini].strip() == "" and palabra(toks[j])[0].isalpha():
            t = toks[j]
            w, p0 = palabra(t)
            if w in DETS: util = True; ini = p0; break
            elif clave_lexico(t) and t.pos_ in ("NOUN", "ADJ", "PROPN"): util = True
            elif t.pos_ != "ADJ": break
            ini = t.idx; j -= 1
        if ini != n.ini and util:
            n.ini, n.texto = ini, texto[ini:n.fin]
    return nombres

Modelo spaCy: es_core_news_lg


## 4. Estructura del diálogo: ¿quién habla y a quién?

En narrativa, buena parte de las menciones son pronombres de **primera y segunda persona dentro del diálogo** (*yo, tú, vos, usted*): en el capítulo 13 de *Sab*, *yo/tú/vos/ti/mí* son una fracción enorme de las menciones. Un sistema de correferencia pensado para noticias no sabe resolverlos, porque su antecedente no es un SN anterior, sino **el hablante o el interlocutor del turno**. Esa información está en la tipografía del texto, así que la modelamos explícitamente:

* Un párrafo que empieza por guion (`-`) es un **turno nuevo**. Dentro del párrafo, los guiones alternan **habla** y **acotación del narrador** (`-¡Pobre Sab! -exclamó Teresa-, ¡cuánto…`). Un párrafo que empieza por `»` **continúa** el habla del mismo hablante.
* El **hablante** de un turno es el sujeto de la acotación (*exclamó **Teresa***). Si no hay acotación o no tiene sujeto, se asume **alternancia** (habla el interlocutor del turno anterior).
* El **interlocutor** es el hablante del turno anterior; un vocativo (nombre propio entre comas) lo corrige.
* Dentro del habla, *yo/mi* → hablante; *tú/vos/usted/tu/vuestro* → interlocutor; *él/ella* → un tercero (se penaliza al hablante y al interlocutor).

In [ ]:
RE_DASH = re.compile(r"(?<![\wÀ-ÿ])[-—–]|[-—–](?![\wÀ-ÿ])")

def segmenta(texto):
    """Devuelve (es_dialogo, turno_nuevo, modo): modo[i] = 'h' (habla) o 'n' (narración) para cada carácter."""
    modo = ["n"] * len(texto)
    ini_guion = re.match(r"\s*[-—–]", texto) is not None
    ini_comilla = re.match(r"\s*[»«]", texto) is not None
    if ini_guion or ini_comilla:
        estado, prev = "h", 0
        guiones = [m.start() for m in RE_DASH.finditer(texto)]
        if ini_guion: guiones = guiones[1:]          # el primero abre el habla
        for g in guiones + [len(texto)]:
            for i in range(prev, g): modo[i] = estado
            estado = "n" if estado == "h" else "h"; prev = g
        return True, ini_guion, modo
    # comillas angulares dentro de la narración: «...» es habla
    for m in re.finditer(r"«(.*?)(?:»|$)", texto, flags=re.S):
        for i in range(m.start(), m.end()): modo[i] = "h"
    return False, False, modo

## 5. Correferencia: modelo de saliencia

La resolución se hace **de izquierda a derecha** manteniendo un **estado del discurso** (`Estado`) que contiene:

* la **saliencia** de cada personaje: cada mención suma peso (más si es sujeto o nombre propio) y la saliencia **decae** a la mitad con cada oración y un 40 % con cada párrafo — la idea de Lappin & Leass (1994) y de la teoría de centrado, aplicada a un reparto cerrado de 21 personajes;
* el **hablante** y el **interlocutor** actuales;
* los personajes ya **mencionados** en el capítulo (para decidir si un SN indefinido es una *primera presentación*).

Para cada mención se puntúa cada personaje candidato y se elige el mejor si supera un umbral (si ninguno lo supera, la mención **no se anota**: preferimos precisión a cobertura cuando hay duda).

| Tipo de mención | Puntuación del candidato *c* |
|---|---|
| Nombre propio | fijo (el gazetario) |
| SN con descriptor (*el mulato, el extranjero*) | `2·peso_descriptor(c) + saliencia(c)` |
| SN genérico (*ese hombre, aquel infeliz*) | `saliencia(c)` filtrando por género |
| Parentesco (*mi padre, su hijo, padre de Carlota*) | la tabla `KIN` da `relación(poseedor) → c`; el poseedor es el hablante (*mi*), el interlocutor (*tu*), un personaje saliente (*su*) o el nombre explícito |
| Pronombre 1.ª / 2.ª persona | hablante / interlocutor del turno |
| Pronombre 3.ª persona | `saliencia(c)` filtrando por género; se penaliza al hablante y al interlocutor si estamos en habla |

In [ ]:
class Estado:
    def __init__(self):
        self.sal = collections.defaultdict(float)
        self.hablante = None; self.interloc = None
        self.mencionados = set()
        self.turnos = []                    # participantes del diálogo, el más reciente primero
        self.ultima_oracion = None
    def decae(self, f):
        for c in list(self.sal): self.sal[c] *= f
    def nuevo_parrafo(self):
        self.decae(0.6); self.ultima_oracion = None
    def avanza_oracion(self, o):
        if self.ultima_oracion is not None and o != self.ultima_oracion:
            self.decae(0.55 ** min(o - self.ultima_oracion, 3))
        self.ultima_oracion = o
    def suma(self, c, w):
        self.sal[c] += w; self.mencionados.add(c)

def compatible(c, genero):
    return genero is None or GEN[c] == genero

def mejores(sc, genero, minimo):
    """Devuelve el personaje con mayor puntuación (compatible en género) si supera `minimo`."""
    sc = {c: s for c, s in sc.items() if compatible(c, genero) and c not in PLURAL}
    if not sc: return None
    c = max(sc, key=sc.get)
    return c if sc[c] >= minimo else None

def posibles_poseedores(m, st):
    if m.pos_de: return {m.pos_de: 3.0}
    if m.det == "pos_mi": return {st.hablante: 3.0} if st.hablante else {}
    if m.det == "pos_tu": return {st.interloc: 3.0} if st.interloc else {}
    return dict(st.sal)                                   # su / el: cualquiera de los personajes salientes

def resuelve(m, st, en_habla):
    """Función pura: devuelve la lista de personajes (ids 1..21) a los que remite la mención, o None."""
    if m.tipo == "nombre":
        return [next(iter(m.cand))]
    if m.tipo == "colectivo":
        top = sorted((c for c in st.sal if c not in PLURAL), key=lambda c: -st.sal[c])[:2]
        return top if len(top) == 2 and st.sal[top[1]] > .3 else None
    if m.tipo == "pron":
        if m.persona == 1:
            return [st.hablante] if st.hablante else None
        if m.persona == 2:
            return [st.interloc] if st.interloc else None
        sc = {}
        for c in set(st.sal) | set(PRIOR):
            s = st.sal.get(c, 0) + .05 * PRIOR.get(c, 0)
            if en_habla and c in (st.hablante, st.interloc): s *= .15
            sc[c] = s
        c = mejores(sc, m.genero, .02)
        return [c] if c else None
    # --- sintagma nominal ---
    fuerte = max(m.cand.values(), default=0)
    if m.det == "none":
        # sin determinante solo anotamos el VOCATIVO dentro del habla («señora», «buen amigo», «pobre joven»),
        # no los atributos («soy mulato», «condición de esclavo»)
        if en_habla and m.prev not in PREP_COP and (m.sig == "" or m.sig in ",.;:!?-—"):
            return [st.interloc] if st.interloc and compatible(st.interloc, m.genero) else None
        return None
    sc = {c: 2 * w + st.sal.get(c, 0) for c, w in m.cand.items()}
    rel = REL.get(m.lema)
    if rel:
        pos = posibles_poseedores(m, st)
        for (r, p), t in KIN.items():
            if r == rel and p in pos and t != p:
                sc[t] = max(sc.get(t, 0), 1.5 + pos[p])
    if m.det == "indef":
        sc = {c: s for c, s in sc.items() if c not in st.mencionados and m.cand.get(c, 0) >= .6}
        c = mejores(sc, m.genero, 1.2)
        return [c] if c else None
    if not m.cand and not rel and m.lema in GENERICOS:    # genérico anafórico: «ese hombre»
        sc = {c: st.sal.get(c, 0) for c in st.sal}
        c = mejores(sc, m.genero, 1.0)
        return [c] if c else None
    if not m.cand and not rel: return None
    c = mejores(sc, m.genero, 1.2 if fuerte >= .6 or rel else 1.0)
    return [c] if c else None

In [ ]:
PREP_COP = {"de","como","por","para","con","ser","soy","eres","es","era","fue","sea","seré","eran","sin","en","a","entre","tan","muy"}
PRIORIDAD = {"nombre": 0, "np": 1, "colectivo": 2, "pron": 3}

def selecciona(menciones):
    """Elimina solapamientos (no hay anidamiento en la anotación): gana la mención más larga; a igual longitud, la más 'fuerte'."""
    elegidas = []
    for m in sorted(menciones, key=lambda m: (-(m.fin - m.ini), PRIORIDAD[m.tipo], m.ini)):
        if all(m.fin <= e.ini or m.ini >= e.fin for e in elegidas):
            elegidas.append(m)
    return sorted(elegidas, key=lambda m: m.ini)

def procesa_parrafo(texto, doc, st):
    """Detecta y resuelve las menciones de un párrafo actualizando el estado del discurso."""
    es_dialogo, turno_nuevo, modo = segmenta(texto)
    st.nuevo_parrafo()
    # --- menciones candidatas ---
    nombres = extiende_nombres(doc, texto, menciones_nombre(texto))
    ocupado = [(n.ini, n.fin) for n in nombres]
    todas = nombres + menciones_ner(doc, ocupado) + menciones_pron(texto) + menciones_np(doc, texto)
    todas = [m for m in todas if not (m.tipo == "colectivo" and
             any(t.idx == m.fin + 1 and t.pos_ in ("NOUN", "ADJ", "PROPN", "NUM") for t in doc))]
    ms = selecciona(todas)
    # oración de cada mención
    ini_oracion = [(s.start_char, i) for i, s in enumerate(doc.sents)]
    for m in ms:
        m.oracion = max([i for a, i in ini_oracion if a <= m.ini], default=0)
        m.en_habla = modo[m.ini] == "h"
    # --- hablante del turno: sujeto de la acotación del narrador ---
    if turno_nuevo:
        hablante = None
        inserts, i = [], 0
        while i < len(texto):
            if modo[i] == "n":
                j = i
                while j < len(texto) and modo[j] == "n": j += 1
                inserts.append((i, j)); i = j
            else: i += 1
        for a, b in inserts:
            seg = texto[a:b].strip(" -—–,.;:")
            if not VERBOS_HABLA.match(seg): continue
            for m in ms:
                if a <= m.ini < b and m.tipo != "colectivo" and m.persona == 3 and \
                        not re.search(r"\b(?:a|en|con|de|hacia|por|para|ante|sobre|contra)\s+(?:(?:el|la|su|mi|aquel|ese)\s+)?$", texto[a:m.ini]):
                    if m.tipo == "pron":                          # «-dijo ella-»: participante del diálogo del género adecuado
                        cands = [c for c in st.turnos if compatible(c, m.genero) and c != st.hablante]
                        cands += [c for c in st.turnos if compatible(c, m.genero) and c == st.hablante]
                        cands += [c for c in sorted(st.sal, key=lambda c: -st.sal[c]) if compatible(c, m.genero) and c not in cands]
                        r = [cands[0]] if cands else None
                    else:
                        r = resuelve(m, st, False)
                    if r: hablante = r[0]; break
            if hablante: break
        if hablante is None:                                      # alternancia
            hablante = st.interloc
        if hablante is None and st.hablante is None and st.sal:   # primer turno sin pistas: el personaje más saliente habla
            hablante = max(st.sal, key=st.sal.get)
        if hablante and hablante != st.hablante:
            st.interloc, st.hablante = st.hablante, hablante
        if st.interloc is None and st.hablante:                   # interlocutor: el otro personaje más saliente
            otros = {c: s for c, s in st.sal.items() if c != st.hablante and s > .02}
            if otros: st.interloc = max(otros, key=otros.get)
        for c in (st.interloc, st.hablante):
            if c:
                if c in st.turnos: st.turnos.remove(c)
                st.turnos.insert(0, c)
        if st.hablante: st.suma(st.hablante, 1.5)
        if st.interloc: st.suma(st.interloc, 1.0)
    if DEBUG and es_dialogo: print("TURNO", st.hablante, st.interloc, "|", texto[:90].replace("\n"," "))
    ini_habla = next((i for i, x in enumerate(modo) if x == "h"), 0)
    # --- resolución mención a mención ---
    for m in ms:
        st.avanza_oracion(m.oracion)
        r = resuelve(m, st, m.en_habla)
        if r is None and m.tipo == "pron" and m.persona in (1, 2) and st.sal:   # sin hablante conocido: el más saliente
            r = [max(st.sal, key=st.sal.get)]
        m.ref = r
        if r:
            peso = (.5 if m.tipo == "pron" and m.persona < 3 else 1.0) + (.8 if m.sujeto else 0) + (.5 if m.tipo == "nombre" else 0)
            for c in r: st.suma(c, peso)
            # un nombre dentro del habla que no es el hablante funciona como vocativo
            if m.tipo == "nombre" and m.en_habla and r[0] != st.hablante and ((st.interloc is None and m.ini - ini_habla < 25) or
                    (re.search(r"(?:^|[,;:¡¿\-—])\s*$", texto[max(0, m.ini - 4):m.ini]) and re.match(r"\s*[,;:!?.\-—]", texto[m.fin:m.fin + 3]))):
                st.interloc = r[0]
    return [m for m in ms if m.ref]

## 6. Escritura del XML anotado

Para cada mención se calcula la posición exacta en el XML original (gracias al mapa de la sección 2) y se insertan `<rs type="person" ref="#prXXX">` y `</rs>`. **Salvaguardas:**
* Si una mención empezase dentro de un elemento (p. ej. `<emph>`) y terminase fuera, se **descarta** (insertarla rompería el XML).
* Al final se **parsea el fichero completo con lxml** para comprobar que sigue bien formado.

In [ ]:
def anota_novela(xml, solo=None, verbose=True):
    inserciones, n_men, descartadas = [], 0, 0
    contador = collections.Counter()
    for n, (a, b) in caps.items():
        if solo and n not in solo: continue
        st, pars = Estado(), []
        for pi, pf in parrafos(xml, a, b):
            pars.append((pi,) + aplana(xml[pi:pf])[:4])
        docs = nlp.pipe([p[1] for p in pars], batch_size=32)
        for (pi, texto, ini, fin, elem), doc in zip(pars, docs):
            for m in procesa_parrafo(texto, doc, st):
                if elem[m.ini] != elem[m.fin - 1]:
                    descartadas += 1; continue
                ref = " ".join("#" + ids_oficiales[c - 1] for c in m.ref)
                inserciones.append((pi + ini[m.ini], 1, f'<rs type="person" ref="{ref}">'))
                inserciones.append((pi + fin[m.fin - 1], 0, "</rs>"))
                n_men += 1; contador[tuple(m.ref)] += 1
    salida = xml
    for pos, orden, cad in sorted(inserciones, reverse=True):
        salida = salida[:pos] + cad + salida[pos:]
    if verbose: print(f"{n_men} menciones anotadas ({descartadas} descartadas por romper el XML)")
    return salida, contador

## 7. Evaluación con los capítulos anotados a mano (1 y 13)

Comparamos nuestra anotación con el *gold standard* de los capítulos 1 y 13, párrafo a párrafo (se comprueba que el texto de ambos coincide). Métricas:

* **Detección (estricta):** la mención coincide exactamente en límites. P, R y F1.
* **Detección (solapamiento):** basta con que los límites se solapen (más indulgente con «el joven» frente a «el joven de hermosa presencia»).
* **Detección + personaje (estricta):** además coincide el `ref` (esta es la métrica que más se parece a lo que mide la rúbrica).
* **Exactitud de la correferencia:** de las menciones bien delimitadas, qué proporción tiene el `ref` correcto.

In [ ]:
def menciones_por_parrafo(xml, ini=0, fin=None):
    """Por párrafo: (texto sin espacios, {(ini, fin): {ids}}). Se ignoran los espacios al medir posiciones,
    porque el gold y el original pueden diferir en saltos de línea."""
    out = []
    for pi, pf in parrafos(xml, ini, len(xml) if fin is None else fin):
        texto, _, _, _, spans = aplana(xml[pi:pf])
        sin = [i for i, ch in enumerate(texto) if not ch.isspace()]
        pos = {}                                   # índice original -> índice sin espacios
        for k, i in enumerate(sin): pos[i] = k
        cuenta = lambda x: sum(1 for ch in texto[:x] if not ch.isspace())
        out.append(("".join(texto.split()),
                    {(cuenta(s), cuenta(e)): frozenset(r.replace("#", "").split()) for s, e, r in spans}))
    return out

def evalua(salida_xml, num_cap, verbose=True):
    gold = menciones_por_parrafo(leer(URL_GOLD[num_cap]))
    a, b = caps[num_cap]
    pred = menciones_por_parrafo(salida_xml, *next((x, y) for n, x, y in capitulos(salida_xml) if n == num_cap))
    assert len(gold) == len(pred), f"cap {num_cap}: {len(gold)} párrafos gold vs {len(pred)} predichos"
    desalineados = sum(1 for (tg, _), (tp, _) in zip(gold, pred) if tg != tp)   # párrafos con retoques en el gold
    tpE = tpO = tpR = nP = nG = 0
    for (tg, g), (tp, p) in zip(gold, pred):
        if tg != tp:       # el gold tiene retoques de texto (¡ añadidos, palabras cambiadas): reproyectamos sus posiciones
            import difflib
            mapa = {}
            for a, c, n in difflib.SequenceMatcher(None, tg, tp, autojunk=False).get_matching_blocks():
                for k in range(n): mapa[a + k] = c + k
            def sig(x):
                while x < len(tg) and x not in mapa: x += 1
                return mapa.get(x, len(tp))
            def ant(x):
                while x >= 0 and x not in mapa: x -= 1
                return mapa.get(x, -1)
            g = {(sig(s), ant(e - 1) + 1): r for (s, e), r in g.items() if sig(s) < ant(e - 1) + 1}
        nP += len(p); nG += len(g)
        for sp, rp in p.items():
            if sp in g:
                tpE += 1; tpR += (g[sp] == rp)
            if any(not (sp[1] <= sg[0] or sp[0] >= sg[1]) for sg in g): tpO += 1
    pr = lambda tp, n: tp / n if n else 0
    f1 = lambda p, r: 2 * p * r / (p + r) if p + r else 0
    res = {
        "párrafos desalineados": desalineados,
        "menciones gold": nG, "menciones predichas": nP,
        "Detección estricta   P/R/F1": (pr(tpE, nP), pr(tpE, nG), f1(pr(tpE, nP), pr(tpE, nG))),
        "Detección solapada   P/R/F1": (pr(tpO, nP), pr(tpO, nG), f1(pr(tpO, nP), pr(tpO, nG))),
        "Detección+personaje  P/R/F1": (pr(tpR, nP), pr(tpR, nG), f1(pr(tpR, nP), pr(tpR, nG))),
        "Exactitud del personaje (sobre bien delimitadas)": pr(tpR, tpE),
    }
    if verbose:
        print(f"=== Capítulo {num_cap} ===")
        for k, v in res.items():
            print(f"  {k:<50}", tuple(round(x, 3) for x in v) if isinstance(v, tuple) else (round(v, 3) if isinstance(v, float) else v))
    return res, gold, pred

## 8. Ejecución sobre la novela completa

In [ ]:
SALIDA, contador = anota_novela(XML_NOVELA)
etree.fromstring(SALIDA.encode("utf-8"))        # comprueba que el XML sigue bien formado
with open(FICHERO_SALIDA, "w", encoding="utf-8") as f:
    f.write(SALIDA)
print("Guardado:", FICHERO_SALIDA)

2617 menciones anotadas (0 descartadas por romper el XML)
Guardado: SPA1021_GomezDeAvellaneda_Sab_AnotadoUrbánMartínez_PérezFerre_NiculescuRuso_PérezMonpeán.xml


In [ ]:
for n in (1, 13):
    evalua(SALIDA, n)

=== Capítulo 1 ===
  párrafos desalineados                              0
  menciones gold                                     137
  menciones predichas                                139
  Detección estricta   P/R/F1                        (0.777, 0.788, 0.783)
  Detección solapada   P/R/F1                        (0.863, 0.876, 0.87)
  Detección+personaje  P/R/F1                        (0.698, 0.708, 0.703)
  Exactitud del personaje (sobre bien delimitadas)   0.898
=== Capítulo 13 ===
  párrafos desalineados                              15
  menciones gold                                     328
  menciones predichas                                343
  Detección estricta   P/R/F1                        (0.837, 0.875, 0.855)
  Detección solapada   P/R/F1                        (0.883, 0.924, 0.903)
  Detección+personaje  P/R/F1                        (0.767, 0.802, 0.784)
  Exactitud del personaje (sobre bien delimitadas)   0.916


## 9. Discusión: resultados, limitaciones y mejoras

**Resultados** (ejecución de referencia, evaluados contra los capítulos anotados a mano):

| | Detección estricta (F1) | Detección solapada (F1) | Detección + personaje (F1) | Exactitud del personaje |
|---|---|---|---|---|
| Capítulo 1 | 0,78 | 0,87 | 0,70 | 0,90 |
| Capítulo 13 | 0,86 | 0,90 | 0,78 | 0,92 |

**Lectura de los resultados.**
* Cuando la mención está bien delimitada, el personaje asignado es correcto en ≈ 90 % de los casos: el modelo de saliencia + estructura de diálogo funciona. La mayor parte del error restante es de **delimitación** (la diferencia entre «solapada» y «estricta») y de **cobertura de SN descriptivos**.
* El capítulo 13 sale mejor porque es casi todo diálogo entre dos personajes con acotaciones explícitas (*-exclamó Teresa-*): ahí la alternancia de turnos resuelve *yo/tú/vos* con muy poco error. El capítulo 1 tiene mucha **descripción**, con referencias por perífrasis (*el extranjero*, *el recién llegado*, *aquel a quien se dirigían estas palabras*) más difíciles de enlazar sin comprensión del texto.

**Honestidad metodológica.** Las tablas de descriptores y de parentesco se diseñaron mirando la lista de personajes y los capítulos 1 y 13, de modo que **esos dos capítulos no son una medida independiente** de generalización. Para estimar el rendimiento en el resto de la novela habría que anotar a mano un capítulo más (p. ej. el 6) y evaluar sin tocar nada.

**Errores típicos que quedan:**
1. **Genéricos con referente variable**: *el esclavo*, *el infeliz* pueden ser un personaje o «el esclavo» en general (valor genérico). Habría que distinguir lo genérico de lo específico.
2. **Aposiciones**: *su hermano don Carlos* → el gold a veces anota dos menciones y a veces una; nosotros siempre fusionamos.
3. **Metáforas y epítetos**: *ese ángel de tus ilusiones*, *ese ídolo indigno*. Requieren un modelo de lengua capaz de captar que designan a una persona.
4. **Oraciones de relativo** dentro de la mención (*la pobre vieja que me llama su hijo*) y **formas de tratamiento** (*su merced*).
5. **Errores del etiquetador de spaCy** en texto del siglo XIX con signos de diálogo pegados a las palabras (*«-Mi»* → `DET`/`PROPN`); lo mitigamos con listas cerradas de determinantes, pero no del todo.
6. **Errores en cascada del diálogo**: si se asigna mal un hablante, la alternancia propaga el error varios turnos.

**Líneas de mejora.**
* Sustituir las reglas de SN por un **clasificador de menciones** (p. ej. un modelo de tipo BERT/`roberta-base-bne` ajustado con los capítulos anotados, tratando la detección como etiquetado de secuencias BIO) y mantener el modelo de saliencia como *reranker* de la correferencia.
* Usar un LLM como **resolutor de última instancia** solo para las menciones de baja confianza (las que hoy se descartan), con ventana de contexto y la lista de personajes en el *prompt*.
* Aprender los pesos de saliencia (hoy fijados a mano) con un modelo log-lineal entrenado sobre el gold.
* Detección automática de **parentesco** a partir de apariciones tipo *«hija de don Carlos»* para no depender de la tabla `KIN` escrita a mano.
* Anotar más capítulos y usarlos como conjunto de test ciego.